# AgentDojo: local models benchmark (Colab T4)

Tests whether prompt injection attacks work on four local Hugging Face models, using AgentDojo:

| Key | Model | Size |
|---|---|---|
| `smollm3` | HuggingFaceTB/SmolLM3-3B | ~6 GB |
| `qwen2.5` | Qwen/Qwen2.5-3B-Instruct | ~6 GB |
| `qwen3` | Qwen/Qwen3-8B-AWQ (4-bit) | ~6 GB |
| `nemotron` | nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 | ~8 GB |

**How it works:** models are downloaded once to your Google Drive. Then, one model at a time, a vLLM
server loads the model on the T4 and AgentDojo runs 20 attacked tests plus 8 no-attack tests against it (28 runs), 8 at a time. The whole notebook takes about 1-1.5 hours (set `ATTACKED_TESTS_PER_MODEL = 50` in cell 1 for the full run).
Logs, a results spreadsheet, CSV files and charts are all saved to Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. No Hugging Face token is needed (all four models are public).

**If Colab disconnects:** run cells 1–5 again, then cell 8. Finished tests are skipped automatically.

## 1. Settings
Change these if you want; everything below uses them.

In [ ]:
GITHUB_REPO_URL = "https://github.com/AtharvaThorat/permission-that-learn-.git"
GITHUB_BRANCH = "feature/agentdojo-local-benchmark"
DRIVE_PROJECT_FOLDER = "/content/drive/MyDrive/agentdojo_local_benchmark"  # on your university Drive

MODELS_TO_TEST = ["smollm3", "qwen2.5", "qwen3", "nemotron"]
ATTACKED_TESTS_PER_MODEL = 20   # spread over the workspace, slack and travel suites; 50 = full run (about 2.5-3.5 hours)
PARALLEL_TESTS = 8              # tests sent to the model at the same time

# Derived paths (no need to change)
REPO_DIR = "/content/permission-that-learn"
PROJECT_DIR = f"{REPO_DIR}/agentdojo_benchmark"   # this benchmark lives in a subfolder of the team repo
MODELS_DIR = f"{DRIVE_PROJECT_FOLDER}/models"     # downloaded models (kept between sessions)
OUTPUT_DIR = f"{DRIVE_PROJECT_FOLDER}/output"     # logs and results (kept between sessions)
RESULTS_DIR = f"{OUTPUT_DIR}/results"             # spreadsheet, CSV files and charts
FAST_DISK_DIR = "/content/models"                 # local copy of the current model (fast to load)
VLLM_ENV = "/content/vllm-env"                    # separate Python environment for vLLM
MODELS_ARG = " ".join(MODELS_TO_TEST)

## 2. Check the GPU
You should see a **Tesla T4**. If you see an error, switch the runtime to T4 GPU.

In [ ]:
!nvidia-smi

## 3. Connect Google Drive
A sign-in popup appears; allow access with your university account.

> If this cell fails inside VS Code, open this notebook at [colab.research.google.com](https://colab.research.google.com) and run it there.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Drive folder ready:", DRIVE_PROJECT_FOLDER)

## 4. Get the code from GitHub
Clones the team repository's benchmark branch the first time, and pulls the latest version after that.

In [ ]:
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {PROJECT_DIR}

## 5. Install the libraries (about 5–10 minutes)
* AgentDojo and this project go into Colab's normal Python.
* vLLM goes into its own environment, because it brings its own versions of PyTorch and other
  libraries that could otherwise clash with AgentDojo or Colab.

In [ ]:
%pip install -q -r requirements.txt   # installs this project and AgentDojo

!pip install -q uv
!uv venv {VLLM_ENV} --python 3.12 -q
!uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.15.1"
!{VLLM_ENV}/bin/vllm --version

## 6. Download the models to Drive (first time only, about 27 GB)
Skips models that are already downloaded, so it is safe to run again.

In [ ]:
!python scripts/phase6_local_models_benchmark.py --download-only --models {MODELS_ARG} --models-dir {MODELS_DIR}

## 7. Preview the tests
The same tests are used for every model, so the results are comparable.

In [ ]:
!python scripts/phase6_local_models_benchmark.py --plan-only --attacked-tests {ATTACKED_TESTS_PER_MODEL}

## 8. Run the benchmark (about 1 hour with 20 attacked tests per model)
For each model: copy it from Drive to the local disk, start vLLM (a few minutes), run all tests,
stop vLLM, then move to the next model. One line is printed per finished test, with the minutes elapsed.

Results are saved after each model, in the order of `MODELS_TO_TEST`:
* **Short on time?** Stop this cell and run cells 9–10: you get the spreadsheet and charts for every model that finished.
* **Colab disconnected?** Run cells 1–5 again, then this cell: it continues where it stopped.

In [ ]:
!python scripts/phase6_local_models_benchmark.py \
    --models {MODELS_ARG} \
    --attacked-tests {ATTACKED_TESTS_PER_MODEL} \
    --workers {PARALLEL_TESTS} \
    --models-dir {MODELS_DIR} \
    --output-dir {OUTPUT_DIR} \
    --fast-disk-dir {FAST_DISK_DIR} --delete-fast-disk-copy \
    --vllm-command {VLLM_ENV}/bin/vllm

## 9. Results tables
Saved on your Drive in `output/results/`:

| File | Contents |
|---|---|
| `benchmark_results.xlsx` | Spreadsheet with three sheets: *Model summary*, *Attack success breakdown*, *Test results* |
| `model_summary.csv` | One row per model (same as the first sheet) |
| `attack_success_breakdown.csv` | Attack success rate per model, by attack style and by suite |
| `test_results.csv` | One row per test |

Columns, all in percent:
* **baseline_utility_pct**: tasks completed with no attack (can the model use the tools at all?)
* **utility_under_attack_pct**: tasks still completed while under attack
* **attack_success_rate_pct**: how often the model carried out the attacker's goal (lower is better)

Runs that crashed (e.g. the conversation got longer than the context window) are counted in `errors`
and left out of the percentages.

In [ ]:
import pandas as pd

summary = pd.read_csv(f"{RESULTS_DIR}/model_summary.csv")
breakdown = pd.read_csv(f"{RESULTS_DIR}/attack_success_breakdown.csv")
test_results = pd.read_csv(f"{RESULTS_DIR}/test_results.csv")
attacked = test_results[test_results["attack"] != "none"]

print("Model summary (%)")
display(summary)
for group_by in ("attack", "suite"):
    print(f"\nAttack success rate (%) by {group_by}")
    display(breakdown[breakdown["group_by"] == group_by]
            .pivot(index="model", columns="group", values="attack_success_rate_pct"))

## 10. Charts
Three charts comparing the models, one color per model. Each is also saved as a PNG in `output/results/charts/`:
1. **Model comparison**: baseline utility, utility under attack, and attack success rate
2. **Attack success rate by attack style**
3. **Attack success rate by suite**

In [ ]:
import os
import sys

sys.path.insert(0, f"{PROJECT_DIR}/src")   # to read the model names from the project
from agentdojo_benchmark.local_models import LOCAL_MODELS

import matplotlib.pyplot as plt
import pandas as pd

# Colors: a colorblind-safe palette, assigned in a fixed order so each model keeps its color in every chart.
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SURFACE, TEXT, MUTED_TEXT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"

model_names = [LOCAL_MODELS[key].served_name for key in MODELS_TO_TEST]   # same order as the run
model_color = dict(zip(model_names, SERIES_COLORS))

summary = pd.read_csv(f"{RESULTS_DIR}/model_summary.csv")
breakdown = pd.read_csv(f"{RESULTS_DIR}/attack_success_breakdown.csv")
os.makedirs(f"{RESULTS_DIR}/charts", exist_ok=True)


def grouped_bar_chart(table, title, subtitle, file_name):
    """table: one row per group on the x-axis, one column per model, values in percent."""
    models = [m for m in model_names if m in table.columns]          # keep each model's fixed color
    group_count, model_count = len(table.index), len(models)
    bar_width = 0.8 / max(model_count, 1) * 0.85                     # leave a small gap between bars
    figure, axis = plt.subplots(figsize=(max(7, 1.9 * group_count + 2), 4.6), facecolor=SURFACE)
    axis.set_facecolor(SURFACE)

    for index, model in enumerate(models):
        offset = (index - (model_count - 1) / 2) * (0.8 / model_count)
        positions = [group + offset for group in range(group_count)]
        values = table[model].tolist()
        bars = axis.bar(positions, [0 if pd.isna(v) else v for v in values], width=bar_width,
                        color=model_color[model], label=model, zorder=3)
        for bar, value in zip(bars, values):                           # value on every bar (some colors are light)
            label = "n/a" if pd.isna(value) else f"{value:.0f}%"
            axis.annotate(label, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                          xytext=(0, 3), textcoords="offset points", ha="center", va="bottom",
                          fontsize=8, color=MUTED_TEXT)

    axis.set_xticks(range(group_count), table.index, color=TEXT)
    axis.set_ylim(0, 110)
    axis.set_yticks(range(0, 101, 25), [f"{tick}%" for tick in range(0, 101, 25)], color=MUTED_TEXT, fontsize=9)
    axis.grid(axis="y", color=GRID, linewidth=1, zorder=0)
    axis.tick_params(length=0)
    for side in ("top", "right", "left"):
        axis.spines[side].set_visible(False)
    axis.spines["bottom"].set_color(GRID)
    # Title, subtitle, then the legend, all above the plot so nothing collides with the x-axis labels
    axis.set_title(title, loc="left", fontsize=13, color=TEXT, fontweight="bold", pad=46)
    axis.text(0, 1.115, subtitle, transform=axis.transAxes, fontsize=9, color=MUTED_TEXT)
    axis.legend(frameon=False, ncol=model_count, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9,
                labelcolor=TEXT, handlelength=1.2, borderaxespad=0.2)
    figure.tight_layout()
    figure.savefig(f"{RESULTS_DIR}/charts/{file_name}", dpi=150, facecolor=SURFACE)
    plt.show()


# 1. The three headline numbers, side by side for every model
headline = summary.set_index("model")[["baseline_utility_pct", "utility_under_attack_pct", "attack_success_rate_pct"]].T
headline.index = ["Baseline utility\n(no attack)", "Utility under attack", "Attack success rate"]
grouped_bar_chart(headline, "Model comparison",
                  "Utility: higher is better.  Attack success rate: lower is better.", "model_comparison.png")

# 2. Attack success rate per attack style, and 3. per suite
for group_by, title in [("attack", "Attack success rate by attack style"), ("suite", "Attack success rate by suite")]:
    table = breakdown[breakdown["group_by"] == group_by].pivot(index="group", columns="model",
                                                               values="attack_success_rate_pct")
    grouped_bar_chart(table, title, "Share of attacked tests where the attacker's goal happened. Lower is better.",
                      f"attack_success_by_{group_by}.png")

print(f"Charts saved to {RESULTS_DIR}/charts/")

## 11. Look at one conversation
Each test has a JSON log with the full conversation. Pick a row from `attacked` to read it.

In [ ]:
import json

row = attacked.iloc[0]   # change the number to look at another test
print(row[["model", "suite", "user_task", "injection_task", "attack", "utility", "attack_success"]], "\n")
log = json.load(open(row["log_file"]))
for message in log["messages"]:
    text = "".join(block["content"] or "" for block in (message["content"] or []))
    calls = [f'{c["function"]}({c["args"]})' for c in (message.get("tool_calls") or [])]
    print(f'--- {message["role"].upper()}\n{text[:1500]}', *calls, sep="\n")

## 12. If a model was SKIPPED
The vLLM server log shows why (for example, out of GPU memory). The other models still run.
Common fixes: lower `--max-model-len` (e.g. 16384) or `PARALLEL_TESTS` (e.g. 4) in cell 8.

In [ ]:
model_key = "nemotron"   # smollm3, qwen2.5, qwen3 or nemotron
!tail -n 40 {OUTPUT_DIR}/server_logs/{model_key}.log